In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from pytorch_msssim import ms_ssim
import scanpy as sc

In [ ]:
def get_plot_array(x, y, c, vmax=None, vmin=None, size=1.):
    # Create figure without axes/margins
    fig, ax = plt.subplots(figsize=(5.4, 7.5), dpi=300)
    ax.scatter(x, y, c=c, s=size, vmax=vmax, vmin=vmin, cmap='RdBu_r')
    ax.set_axis_off()
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # Render to a NumPy array (RGB)
    fig.canvas.draw()
    rgba_buffer = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close(fig)
    
    # Convert RGBA to RGB and normalize to [0, 1]
    rgb_array = rgba_buffer[:, :, :3] / 255.0
    return rgb_array

In [ ]:
adata_ccc = sc.read_h5ad("./VisiumHD_acts_progeny.h5ad")
adata_ccc_orignal = sc.read_h5ad("./VisiumHD_original_acts_progeny.h5ad")

In [ ]:
adata_ccc_orignal.obs['VEGF_mlm'] = adata_ccc_orignal.obsm['score_mlm'].iloc[:, 10]
adata_ccc.obs['VEGF_mlm'] = adata_ccc.obsm['score_mlm'].iloc[:, 10]
adata_ccc.obs['PI3K_mlm'] = adata_ccc.obsm['score_mlm'].iloc[:, 7]
adata_ccc.obs['Hypoxia_mlm'] = adata_ccc.obsm['score_mlm'].iloc[:, 3]
adata_ccc.obs['JAK-STAT_mlm'] = adata_ccc.obsm['score_mlm'].iloc[:, 4]
adata_ccc.obs['MAPK_mlm'] = adata_ccc.obsm['score_mlm'].iloc[:, 5]
adata_ccc.obs['NFkB_mlm'] = adata_ccc.obsm['score_mlm'].iloc[:, 6]

In [ ]:
adata_ccc

In [ ]:
adata_ccc_orignal

In [ ]:
# Sample Data
x1, y1 = adata_ccc.obsm['spatial'][:, 0], adata_ccc.obsm['spatial'][:, 1]
x2, y2 = adata_ccc_orignal.obsm['spatial'][:, 0], adata_ccc_orignal.obsm['spatial'][:, 1]
# Get arrays
arr1 = get_plot_array(x1, y1, adata_ccc.obs['EGFR_mlm'], size=1.0)
arr2 = get_plot_array(x2, y2, adata_ccc_orignal.obs['EGFR_mlm'], size=1.0, vmin=-6)
print(arr1.shape)

# Convert to PyTorch Tensors (Batch, Channel, Height, Width)
img1 = torch.from_numpy(arr1).permute(2, 0, 1).unsqueeze(0).float()
img2 = torch.from_numpy(arr2).permute(2, 0, 1).unsqueeze(0).float()

In [ ]:
plt.imshow(arr1)

In [ ]:
sc.pl.spatial(adata_ccc_orignal, color='VEGF_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmin=-6, vmax=5)

In [ ]:
marker_list = ['VEGF_mlm', 'PI3K_mlm', 'Hypoxia_mlm', 'JAK-STAT_mlm', 'MAPK_mlm', 'NFkB_mlm']

In [ ]:
ssim_score = ms_ssim(img1, img2, data_range=1.0, win_size=11).item()

In [ ]:
ssim_score

In [ ]:
def ms_ssim_score_pvalue(adata1, adata2, marker_list, p=True, vmin1=None, vmax1=None, vmin2=None, vmax2=None):
    ms_ssim_score = []
    p_value = []
    for i in range(len(marker_list)):
        marker=marker_list[i]
        # Sample Data
        x1, y1 = adata1.obsm['spatial'][:, 0], adata1.obsm['spatial'][:, 1]
        x2, y2 = adata2.obsm['spatial'][:, 0], adata2.obsm['spatial'][:, 1]
        # Get arrays
        arr1 = get_plot_array(x1, y1, adata1.obs[marker])
        arr2 = get_plot_array(x2, y2, adata2.obs[marker], vmin=vmin2[i], vmax=vmax2[i])

        # Convert to PyTorch Tensors (Batch, Channel, Height, Width)
        img1 = torch.from_numpy(arr1).permute(2, 0, 1).unsqueeze(0).float()
        img2 = torch.from_numpy(arr2).permute(2, 0, 1).unsqueeze(0).float()
        
        ssim_score = ms_ssim(img1, img2, data_range=1.0).item()
        ms_ssim_score.append(ssim_score)
        
        if p == True:
            credibility_array = adata2.obs[marker]
            permutation_score = []
            for i in range(1000):
                if i %200==0:
                    print(i)
                shuffled_credibility = np.random.permutation(credibility_array)
                shuffled_arr2 = get_plot_array(x2, y2, shuffled_credibility)
                shuffled_img2 = torch.from_numpy(shuffled_arr2).permute(2, 0, 1).unsqueeze(0).float()
                permutation_score.append(ms_ssim(img1, shuffled_img2, data_range=1.0))
            p_value.append(np.sum(np.array(permutation_score) >= ssim_score)/1000)

    return ms_ssim_score, p_value

In [ ]:
ms_ssim_list, p_value_list = ms_ssim_score_pvalue(adata_ccc, adata_ccc_orignal, marker_list=marker_list, p=False, vmin2=[-6, -2.5, -2, -3, -3, -2], vmax2=[None, None, 3, None, 2.5, 3.5])

In [ ]:
ms_ssim_list

In [ ]:
ms_ssim_list, p_value_list = ms_ssim_score_pvalue(adata_ccc, adata_ccc_orignal, marker_list=marker_list, p=True, vmin2=[-6, -2.5, -2, -3, -3, -2], vmax2=[None, None, 3, None, 2.5, 3.5])

In [ ]:
ms_ssim_list

In [ ]:
p_value_list